In [ ]:
# ============================================================
# 06 — ABLATION: FROZEN vs ROLLING popularity (EB-NeRD)
# Headline finding: frozen popularity (leaks+drifts) 0.568 -> rolling point-in-time 0.773 (+0.205).
# Fully self-contained EB-NeRD notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers polars -q
import os, glob, math, zipfile, numpy as np, polars as pl, datetime as dt, lightgbm as lgb, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
# ---- hardcoded EB-NeRD demo path (fast offline iteration) ----
DEMO = "/kaggle/input/datasets/donbosoc/ebnerd-small"
if not os.path.exists(f"{DEMO}/articles.parquet"):
    DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
print("DEMO:", DEMO)
PREFIX = "eb"
def pfx(x): return f"{PREFIX}:{x}"
def _prefix(col): return pl.concat_str([pl.lit(f"{PREFIX}:"), col.cast(pl.Utf8)])
def parse_split(base, split):
    """Parse EB-NeRD into unified schema: articles / impressions / history."""
    a = pl.read_parquet(f"{base}/articles.parquet")
    articles = a.select(
        article_id=_prefix(pl.col("article_id")),
        title=pl.col("title").fill_null(""),
        abstract=pl.col("subtitle").fill_null(""),
        body=pl.col("body").fill_null("") if "body" in a.columns else pl.lit(""),
        category=pl.col("category_str").fill_null(""),
        published_time=pl.col("published_time"),
    )
    b = pl.read_parquet(f"{base}/{split}/behaviors.parquet")
    cols = b.columns
    impressions = b.select(
        impression_id=pl.col("impression_id"),
        user_id=_prefix(pl.col("user_id")),
        timestamp=pl.col("impression_time"),
        candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
        labels=(pl.col("article_ids_clicked").list.eval(_prefix(pl.element()))
                if "article_ids_clicked" in cols else pl.lit(None)),
        session_id=(pl.col("session_id") if "session_id" in cols else pl.lit(0)),
    )
    h = pl.read_parquet(f"{base}/{split}/history.parquet")
    hist = {u: (arts or []) for u, arts in zip(
        h.select(_prefix(pl.col("user_id")))["user_id"].to_list(),
        h["article_id_fixed"].list.eval(_prefix(pl.element())).to_list())}
    return articles, impressions, hist
def build_article_luts(articles_df, raw_parquet_path):
    """Build published_time / pageview / category / text lookups from the raw articles."""
    at = pl.read_parquet(raw_parquet_path)
    pub={pfx(r):p for r,p in zip(at["article_id"].to_list(), at["published_time"].to_list())}
    cat={pfx(r):(c or "") for r,c in zip(at["article_id"].to_list(), at["category_str"].to_list())}
    txt={pfx(r):f"{t or ''} {s or ''}".strip() for r,t,s in zip(
         at["article_id"].to_list(), at["title"].to_list(), at["subtitle"].to_list())}
    def col_or_zero(name):
        if name in at.columns:
            return {pfx(r):(v or 0) for r,v in zip(at["article_id"].to_list(), at[name].to_list())}
        return defaultdict(float)
    pv=col_or_zero("total_pageviews"); iv=col_or_zero("total_inviews"); rt=col_or_zero("total_read_time")
    return pub,cat,txt,pv,iv,rt


In [ ]:
art, imp_tr, hist_tr = parse_split(DEMO, "train")
_,   imp_va, hist_va = parse_split(DEMO, "validation")
pub,cat,txt,pv,iv,rt = build_article_luts(art, f"{DEMO}/articles.parquet")
print("articles:", len(pub), "| train imps:", imp_tr.height, "| val imps:", imp_va.height)
from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
_aids = list(txt.keys()); _atxt = [txt[i] if txt[i] else "nyhed" for i in _aids]
emb = minilm.encode(_atxt, batch_size=512, normalize_embeddings=True,
                    convert_to_numpy=True, show_progress_bar=True)
emb_by_id = {_aids[i]: emb[i] for i in range(len(_aids))}
id_to_row = {_aids[i]: i for i in range(len(_aids))}
emb_mat = emb
print("multilingual MiniLM encoded:", emb.shape)
def recency(aid,T,tau=24.0):
    p=pub.get(aid)
    if p is None or T is None: return 0.0
    dh=(T-p).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def mm(x):
    lo,hi=x.min(),x.max();return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)
def user_prof(hist,mh=30):
    ai=hist[-mh:] if hist else []
    cats=[cat.get(x) for x in ai];tot=len([c for c in cats if c])
    cc=Counter(c for c in cats if c);cd={k:v/tot for k,v in cc.items()} if tot else {}
    hv=[emb_by_id[x] for x in ai if x in emb_by_id]
    um=np.mean(hv,0) if hv else None
    if um is not None: um=um/(np.linalg.norm(um)+1e-9)
    return cd,um,hv
def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def mrr_i(s,lb):
    o=np.argsort(-s)
    for rank,idx in enumerate(o,1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def ndcg_i(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    ideal=np.sort(lb)[::-1][:k]
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(ideal))
    return float(dcg/idcg) if idcg>0 else 0.0
def iter_impressions(imps_df):
    for row in imps_df.iter_rows(named=True):
        cand=row["candidate_ids"]; labs=row["labels"]; T=row["timestamp"]
        if not cand or not labs: continue
        y=np.array([1 if c in set(labs) else 0 for c in cand])
        if y.sum()==0: continue
        yield row["user_id"], T, cand, y


In [ ]:
# frozen popularity = total clicks over TRAIN, reused for every impression
frozen_pop=Counter()
for row in imp_tr.iter_rows(named=True):
    for c in (row["labels"] or []): frozen_pop[c]+=1
# rolling click index (point-in-time)
click_ev=defaultdict(list)
for imps in (imp_tr,imp_va):
    for row in imps.iter_rows(named=True):
        T=row["timestamp"]
        for c in (row["labels"] or []): click_ev[c].append(T)
for k in click_ev: click_ev[k].sort()
def pit(aid,T): 
    tl=click_ev.get(aid); return bisect_left(tl,T) if tl else 0

def build_simple(imps,hl,mode):
    rows=list(iter_impressions(imps));X=[];y=[];g=[]
    for uid,T,cand,labs in rows:
        cd,um,hv=user_prof(hl.get(uid,[]));m=len(cand);F=[]
        for i,c in enumerate(cand):
            rec=recency(c,T)
            p= pit(c,T) if mode=="rolling" else frozen_pop.get(c,0)
            cm=cd.get(cat.get(c,""),0.0)
            F.append([rec,p,cm,m])
        F=np.array(F,float);F[:,0]=mm(F[:,0]);F[:,1]=mm(F[:,1])
        X.append(F);y.append(labs);g.append(m)
    return np.vstack(X),np.concatenate(y),g

res={}
for mode in ("frozen","rolling"):
    Xtr,ytr,gtr=build_simple(imp_tr,hist_tr,mode)
    Xva,yva,gva=build_simple(imp_va,hist_va,mode)
    rk=lgb.LGBMRanker(objective="lambdarank",n_estimators=400,learning_rate=0.03,
                      num_leaves=31,min_child_samples=50,verbose=-1)
    rk.fit(Xtr,ytr,group=gtr)
    sc=rk.predict(Xva);pos=0;aucs=[]
    for gi in gva:
        a=auc_i(sc[pos:pos+gi],yva[pos:pos+gi]);pos+=gi
        if a is not None: aucs.append(a)
    res[mode]=np.mean(aucs)
print("=== EB-NeRD frozen vs rolling popularity ===")
print(f"  frozen  AUC: {res['frozen']:.4f}")
print(f"  rolling AUC: {res['rolling']:.4f}")
print(f"  delta      : {res['rolling']-res['frozen']:+.4f}")
print("Frozen popularity leaks AND drifts; rolling point-in-time recovers performance.")
